> Adapted from *Dive into Deep Learning*, section [Learning Rate Scheduling](https://d2l.ai/chapter_optimization/lr-scheduler.html). Not affiliated with or endorsed by the book's authors.
> Source, authors, licenses and changes: [Attribution and changes](#Attribution-and-changes), at the end of this notebook.

Previous: [Section 12.10: Adam](12.10_adam.ipynb)

# Learning Rate Scheduling

So far we primarily focused on optimization *algorithms* for how to update the weight vectors rather than on the *rate* at which they are being updated. Nonetheless, adjusting the learning rate is often just as important as the actual algorithm. There are a number of aspects to consider:

* Most obviously the *magnitude* of the learning rate matters. If it is too large, optimization diverges, if it is too small, it takes too long to train or we end up with a suboptimal result. We saw previously that the condition number of the problem matters (see e.g., [Section 12.6](https://d2l.ai/chapter_optimization/momentum.html#sec-momentum) for details). Intuitively it is the ratio of the amount of change in the least sensitive direction vs. the most sensitive one.
* Secondly, the rate of decay is just as important. If the learning rate remains large we may simply end up bouncing around the minimum and thus not reach optimality. [Section 12.5](https://d2l.ai/chapter_optimization/minibatch-sgd.html#sec-minibatch-sgd) discussed this in some detail and we analyzed performance guarantees in [Section 12.4](https://d2l.ai/chapter_optimization/sgd.html#sec-sgd). In short, we want the rate to decay, but probably more slowly than $\mathcal{O}(t^{-\frac{1}{2}})$ which would be a good choice for convex problems.
* Another aspect that is equally important is *initialization*. This pertains both to how the parameters are set initially (review [Section 5.4](https://d2l.ai/chapter_multilayer-perceptrons/numerical-stability-and-init.html#sec-numerical-stability) for details) and also how they evolve initially. This goes under the moniker of *warmup*, i.e., how rapidly we start moving towards the solution initially. Large steps in the beginning might not be beneficial, in particular since the initial set of parameters is random. The initial update directions might be quite meaningless, too.
* Lastly, there are a number of optimization variants that perform cyclical learning rate adjustment. This is beyond the scope of the current chapter. We recommend the reader to review details in Izmailov et al. (2018), e.g., how to obtain better solutions by averaging over an entire *path* of parameters.

Given the fact that there is a lot of detail needed to manage learning rates, most deep learning frameworks have tools to deal with this automatically. In the current chapter we will review the effects that different schedules have on accuracy and also show how this can be managed efficiently via a *learning rate scheduler*.

## Toy Problem

We begin with a toy problem that is cheap enough to compute easily, yet sufficiently nontrivial to illustrate some of the key aspects. For that we pick a slightly modernized version of LeNet (`relu` instead of `sigmoid` activation, MaxPooling rather than AveragePooling), as applied to Fashion-MNIST. Moreover, we hybridize the network for performance. Since most of the code is standard we just introduce the basics without further detailed discussion. See [Chapter 7](https://d2l.ai/chapter_convolutional-neural-networks/index.html#chap-cnn) for a refresher as needed.

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> In the browser this notebook trains on fewer images than the book, so that its five training runs finish: the first code cell loads 2,000 training images and 2,000 test images (1,000 of each in Pyodide) with `d2l.browser_examples`, against the book's 60,000 and 10,000, and keeps the book's 30 epochs, batch size and learning rates. The curves show less training and the accuracies are lower than the book's. Replace `num_examples` by `None` to use all the images, which takes about an hour.

In [ ]:
import math
import numpy as np
from autograd.tracer import getval

import divelet_ml as d2l
from divelet_ml import nn, optim
from divelet_ml.optim import lr_scheduler

def net_fn():
    model = nn.Sequential(
        nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Conv2d(6, 16, kernel_size=5), nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Flatten(),
        nn.Linear(16 * 5 * 5, 120), nn.ReLU(),
        nn.Linear(120, 84), nn.ReLU(),
        nn.Linear(84, 10))

    return model

loss = nn.cross_entropy

batch_size = 256
num_examples = d2l.browser_examples(2000, pyodide=1000, xeus=2000)
train_iter, test_iter = d2l.load_data_fashion_mnist(
    batch_size=batch_size, num_train=num_examples, num_val=num_examples)

# The code is almost identical to `d2l.train_ch6` defined in the
# lenet section of chapter convolutional neural networks
def train(net, train_iter, test_iter, num_epochs, loss, trainer,
          scheduler=None):
    animator = d2l.Animator(xlabel='epoch', xlim=[0, num_epochs],
                            legend=['train loss', 'train acc', 'test acc'])

    for epoch in range(num_epochs):
        metric = d2l.Accumulator(3)  # train_loss, train_acc, num_examples
        for i, (X, y) in enumerate(train_iter):
            net.train()
            seen = {}
            def objective():
                seen['y_hat'] = net(X)
                seen['l'] = loss(seen['y_hat'], y)
                return seen['l']
            trainer.step(d2l.param_grads(objective, net, trainer.params))
            metric.add(float(getval(seen['l'])) * X.shape[0],
                       d2l.accuracy(getval(seen['y_hat']), y), X.shape[0])
            train_loss = metric[0] / metric[2]
            train_acc = metric[1] / metric[2]
            if (i + 1) % 50 == 0:
                animator.add(epoch + i / len(train_iter),
                             (train_loss, train_acc, None))

        test_acc = d2l.evaluate_accuracy_gpu(net, test_iter)
        animator.add(epoch+1, (None, None, test_acc))

        if scheduler:
            if isinstance(scheduler, lr_scheduler.MultiStepLR):
                # Using the built-in scheduler
                scheduler.step()
            else:
                # Using custom defined scheduler
                for param_group in trainer.param_groups:
                    param_group['lr'] = scheduler(epoch)

    print(f'train loss {train_loss:.3f}, train acc {train_acc:.3f}, '
          f'test acc {test_acc:.3f}')

Let's have a look at what happens if we invoke this algorithm with default settings, such as a learning rate of $0.3$ and train for $30$ iterations. Note how the training accuracy keeps on increasing while progress in terms of test accuracy stalls beyond a point. The gap between both curves indicates overfitting.

In [ ]:
lr, num_epochs = 0.3, 30
net = net_fn()
trainer = optim.SGD(net.parameters(), lr=lr)
train(net, train_iter, test_iter, num_epochs, loss, trainer)

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> **Full run, for comparison.** In JupyterLite, the cell above trains less than the book does (see the note before it). This figure is what it gives with 30 epochs on all 60,000 training images and 10,000 test images, as in the book: computed with CPython, not in the browser, from a random start (seed 0), so a run of your own will differ in the details. Final values: train loss 0.163, train acc 0.937, test acc 0.884.

Training curves of the full run. Final values: train loss 0.163, train acc 0.937, test acc 0.884.

## Schedulers

One way of adjusting the learning rate is to set it explicitly at each step. This is conveniently achieved by the `set_learning_rate` method. We could adjust it downward after every epoch (or even after every minibatch), e.g., in a dynamic manner in response to how optimization is progressing.

In [ ]:
lr = 0.1
trainer.param_groups[0]["lr"] = lr
print(f'learning rate is now {trainer.param_groups[0]["lr"]:.2f}')

More generally we want to define a scheduler. When invoked with the number of updates it returns the appropriate value of the learning rate. Let's define a simple one that sets the learning rate to $\eta = \eta_0 (t + 1)^{-\frac{1}{2}}$.

In [ ]:
class SquareRootScheduler:
    def __init__(self, lr=0.1):
        self.lr = lr

    def __call__(self, num_update):
        return self.lr * pow(num_update + 1.0, -0.5)

Let's plot its behavior over a range of values.

In [ ]:
scheduler = SquareRootScheduler(lr=0.1)
d2l.plot(np.arange(num_epochs), [scheduler(t) for t in range(num_epochs)])

Now let's see how this plays out for training on Fashion-MNIST. We simply provide the scheduler as an additional argument to the training algorithm.

In [ ]:
net = net_fn()
trainer = optim.SGD(net.parameters(), lr)
train(net, train_iter, test_iter, num_epochs, loss, trainer,
      scheduler)

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> **Full run, for comparison.** In JupyterLite, the cell above trains less than the book does (see the note before it). This figure is what it gives with 30 epochs on all 60,000 training images and 10,000 test images, as in the book: computed with CPython, not in the browser, from a random start (seed 0), so a run of your own will differ in the details. Final values: train loss 0.273, train acc 0.900, test acc 0.883.

Training curves of the full run. Final values: train loss 0.273, train acc 0.900, test acc 0.883.

This worked quite a bit better than previously. Two things stand out: the curve was rather more smooth than previously. Secondly, there was less overfitting. Unfortunately it is not a well-resolved question as to why certain strategies lead to less overfitting in *theory*. There is some argument that a smaller stepsize will lead to parameters that are closer to zero and thus simpler. However, this does not explain the phenomenon entirely since we do not really stop early but simply reduce the learning rate gently.

## Policies

While we cannot possibly cover the entire variety of learning rate schedulers, we attempt to give a brief overview of popular policies below. Common choices are polynomial decay and piecewise constant schedules. Beyond that, cosine learning rate schedules have been found to work well empirically on some problems. Lastly, on some problems it is beneficial to warm up the optimizer prior to using large learning rates.

### Factor Scheduler

One alternative to a polynomial decay would be a multiplicative one, that is $\eta_{t+1} \leftarrow \eta_t \cdot \alpha$ for $\alpha \in (0, 1)$. To prevent the learning rate from decaying beyond a reasonable lower bound the update equation is often modified to $\eta_{t+1} \leftarrow \mathop{\mathrm{max}}(\eta_{\mathrm{min}}, \eta_t \cdot \alpha)$.

In [ ]:
class FactorScheduler:
    def __init__(self, factor=1, stop_factor_lr=1e-7, base_lr=0.1):
        self.factor = factor
        self.stop_factor_lr = stop_factor_lr
        self.base_lr = base_lr

    def __call__(self, num_update):
        self.base_lr = max(self.stop_factor_lr, self.base_lr * self.factor)
        return self.base_lr

scheduler = FactorScheduler(factor=0.9, stop_factor_lr=1e-2, base_lr=2.0)
d2l.plot(np.arange(50), [scheduler(t) for t in range(50)])

This can also be accomplished by a built-in scheduler in MXNet via the `lr_scheduler.FactorScheduler` object. It takes a few more parameters, such as warmup period, warmup mode (linear or constant), the maximum number of desired updates, etc.; Going forward we will use the built-in schedulers as appropriate and only explain their functionality here. As illustrated, it is fairly straightforward to build your own scheduler if needed.

### Multi Factor Scheduler

A common strategy for training deep networks is to keep the learning rate piecewise constant and to decrease it by a given amount every so often. That is, given a set of times when to decrease the rate, such as $s = \{5, 10, 20\}$ decrease $\eta_{t+1} \leftarrow \eta_t \cdot \alpha$ whenever $t \in s$. Assuming that the values are halved at each step we can implement this as follows.

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> The book has a `FactorScheduler` of its own in the cell above and mentions a built-in one of MXNet; divelet-ml has the built-in `MultiStepLR` of PyTorch (`divelet_ml.optim.lr_scheduler`), which the next cell uses.

In [ ]:
net = net_fn()
trainer = optim.SGD(net.parameters(), lr=0.5)
scheduler = lr_scheduler.MultiStepLR(trainer, milestones=[15, 30], gamma=0.5)

def get_lr(trainer, scheduler):
    lr = scheduler.get_last_lr()[0]
    scheduler.step()
    return lr

d2l.plot(np.arange(num_epochs), [get_lr(trainer, scheduler)
                                 for t in range(num_epochs)])

The intuition behind this piecewise constant learning rate schedule is that one lets optimization proceed until a stationary point has been reached in terms of the distribution of weight vectors. Then (and only then) do we decrease the rate such as to obtain a higher quality proxy to a good local minimum. The example below shows how this can produce ever slightly better solutions.

In [ ]:
train(net, train_iter, test_iter, num_epochs, loss, trainer,
      scheduler)

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> **Full run, for comparison.** In JupyterLite, the cell above trains less than the book does (see the note before it). This figure is what it gives with 30 epochs on all 60,000 training images and 10,000 test images, as in the book: computed with CPython, not in the browser, from a random start (seed 0), so a run of your own will differ in the details. Final values: train loss 0.177, train acc 0.934, test acc 0.882.

Training curves of the full run. Final values: train loss 0.177, train acc 0.934, test acc 0.882.

### Cosine Scheduler

A rather perplexing heuristic was proposed by Loshchilov and Hutter (2016). It relies on the observation that we might not want to decrease the learning rate too drastically in the beginning and moreover, that we might want to "refine" the solution in the end using a very small learning rate. This results in a cosine-like schedule with the following functional form for learning rates in the range $t \in [0, T]$.

$$\eta_t = \eta_T + \frac{\eta_0 - \eta_T}{2} \left(1 + \cos(\pi t/T)\right) \tag{12.11.1}$$


Here $\eta_0$ is the initial learning rate, $\eta_T$ is the target rate at time $T$. Furthermore, for $t > T$ we simply pin the value to $\eta_T$ without increasing it again. In the following example, we set the max update step $T = 20$.

In [ ]:
class CosineScheduler:
    def __init__(self, max_update, base_lr=0.01, final_lr=0,
               warmup_steps=0, warmup_begin_lr=0):
        self.base_lr_orig = base_lr
        self.max_update = max_update
        self.final_lr = final_lr
        self.warmup_steps = warmup_steps
        self.warmup_begin_lr = warmup_begin_lr
        self.max_steps = self.max_update - self.warmup_steps

    def get_warmup_lr(self, epoch):
        increase = (self.base_lr_orig - self.warmup_begin_lr) \
                       * float(epoch) / float(self.warmup_steps)
        return self.warmup_begin_lr + increase

    def __call__(self, epoch):
        if epoch < self.warmup_steps:
            return self.get_warmup_lr(epoch)
        if epoch <= self.max_update:
            self.base_lr = self.final_lr + (
                self.base_lr_orig - self.final_lr) * (1 + math.cos(
                math.pi * (epoch - self.warmup_steps) / self.max_steps)) / 2
        return self.base_lr

scheduler = CosineScheduler(max_update=20, base_lr=0.3, final_lr=0.01)
d2l.plot(np.arange(num_epochs), [scheduler(t) for t in range(num_epochs)])

In the context of computer vision this schedule *can* lead to improved results. Note, though, that such improvements are not guaranteed (as can be seen below).

In [ ]:
net = net_fn()
trainer = optim.SGD(net.parameters(), lr=0.3)
train(net, train_iter, test_iter, num_epochs, loss, trainer,
      scheduler)

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> **Full run, for comparison.** In JupyterLite, the cell above trains less than the book does (see the note before it). This figure is what it gives with 30 epochs on all 60,000 training images and 10,000 test images, as in the book: computed with CPython, not in the browser, from a random start (seed 0), so a run of your own will differ in the details. Final values: train loss 0.166, train acc 0.938, test acc 0.900.

Training curves of the full run. Final values: train loss 0.166, train acc 0.938, test acc 0.900.

### Warmup

In some cases initializing the parameters is not sufficient to guarantee a good solution. This is particularly a problem for some advanced network designs that may lead to unstable optimization problems. We could address this by choosing a sufficiently small learning rate to prevent divergence in the beginning. Unfortunately this means that progress is slow. Conversely, a large learning rate initially leads to divergence.

A rather simple fix for this dilemma is to use a warmup period during which the learning rate *increases* to its initial maximum and to cool down the rate until the end of the optimization process. For simplicity one typically uses a linear increase for this purpose. This leads to a schedule of the form indicated below.

In [ ]:
scheduler = CosineScheduler(20, warmup_steps=5, base_lr=0.3, final_lr=0.01)
d2l.plot(np.arange(num_epochs), [scheduler(t) for t in range(num_epochs)])

Note that the network converges better initially (in particular observe the performance during the first 5 epochs).

In [ ]:
net = net_fn()
trainer = optim.SGD(net.parameters(), lr=0.3)
train(net, train_iter, test_iter, num_epochs, loss, trainer,
      scheduler)

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> **Full run, for comparison.** In JupyterLite, the cell above trains less than the book does (see the note before it). This figure is what it gives with 30 epochs on all 60,000 training images and 10,000 test images, as in the book: computed with CPython, not in the browser, from a random start (seed 0), so a run of your own will differ in the details. Final values: train loss 0.259, train acc 0.903, test acc 0.877.

Training curves of the full run. Final values: train loss 0.259, train acc 0.903, test acc 0.877.

Warmup can be applied to any scheduler (not just cosine). For a more detailed discussion of learning rate schedules and many more experiments see also (Gotmare et al., 2018). In particular they find that a warmup phase limits the amount of divergence of parameters in very deep networks. This makes intuitively sense since we would expect significant divergence due to random initialization in those parts of the network that take the most time to make progress in the beginning.

## Summary

* Decreasing the learning rate during training can lead to improved accuracy and (most perplexingly) reduced overfitting of the model.
* A piecewise decrease of the learning rate whenever progress has plateaued is effective in practice. Essentially this ensures that we converge efficiently to a suitable solution and only then reduce the inherent variance of the parameters by reducing the learning rate.
* Cosine schedulers are popular for some computer vision problems. See e.g., [GluonCV](http://gluon-cv.mxnet.io) for details of such a scheduler.
* A warmup period before optimization can prevent divergence.
* Optimization serves multiple purposes in deep learning. Besides minimizing the training objective, different choices of optimization algorithms and learning rate scheduling can lead to rather different amounts of generalization and overfitting on the test set (for the same amount of training error).

## Exercises

1. Experiment with the optimization behavior for a given fixed learning rate. What is the best model you can obtain this way?
1. How does convergence change if you change the exponent of the decrease in the learning rate? Use `PolyScheduler` for your convenience in the experiments.
1. Apply the cosine scheduler to large computer vision problems, e.g., training ImageNet. How does it affect performance relative to other schedulers?
1. How long should warmup last?
1. Can you connect optimization and sampling? Start by using results from Welling and Teh (2011) on Stochastic Gradient Langevin Dynamics.

Discussions of the original section: [book's forum](https://discuss.d2l.ai/t/1080)

Previous: [Section 12.10: Adam](12.10_adam.ipynb)

## Attribution and changes

Adapted from *Dive into Deep Learning*, section
[Learning Rate Scheduling](https://d2l.ai/chapter_optimization/lr-scheduler.html)
(source: [`chapter_optimization/lr-scheduler.md`](https://github.com/d2l-ai/d2l-en/blob/23d7a5aecceee57d1292c56e90cce307f183bb0a/chapter_optimization/lr-scheduler.md)
at commit `23d7a5aeccee`), by Aston Zhang, Zachary C. Lipton, Mu Li, Alexander J. Smola and contributors.
Text: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Code: adapted from the book's sample code, MIT-0.
Not affiliated with or endorsed by the book's authors.

**Changes from the original** (PyTorch edition):
- Code uses NumPy instead of PyTorch, so it runs in JupyterLite (Pyodide and xeus-python).
- `d2l` is divelet-ml's package (`import divelet_ml as d2l`) instead of the book's `d2l.torch`. `%matplotlib inline` is removed: both JupyterLite kernels display plots inline.
- In the browser the notebook trains on fewer images than the book: `d2l.load_data_fashion_mnist` takes `num_train` and `num_val` (not in the book), and `d2l.browser_examples(2000, pyodide=1000, xeus=2000)` gives 2,000 training images and 2,000 test images, and 1,000 of each in Pyodide, against the book's 60,000 and 10,000. The schedules still run for the book's 30 epochs and the batch size and rates are the book's. The curves show less training and the accuracies are lower than the book's; a figure after each training cell shows what the full training gives.
- `nn.Conv2d`, `nn.MaxPool2d`, `nn.Linear` and the like are divelet-ml's `divelet_ml.nn`; `nn.CrossEntropyLoss()` is its function `nn.cross_entropy`. `torch.optim.SGD` is `divelet_ml.optim.SGD` and `lr_scheduler.MultiStepLR` is `divelet_ml.optim.lr_scheduler.MultiStepLR`, which has the `step` and `get_last_lr` of PyTorch's.
- `train` takes no `device` (there is no GPU in a browser) and computes the gradients with `d2l.param_grads`, with `trainer.step(grads)` in place of `trainer.zero_grad()`, `l.backward()` and `trainer.step()`. The book's `trainer.step()` in `get_lr`, which only silences a PyTorch warning, is dropped. The built-in scheduler is recognized with `isinstance` instead of `scheduler.__module__`.
- `d2l.arange` is `np.arange`.
- The data of Fashion-MNIST is read from this site in JupyterLite (see Section 4.2).
- Equations keep the book's numbers. Cross-references to other sections link to the book's site, and citations are written inline (author and year); the book's bibliography lists them.
- The d2lbook framework tabs for MXNet, TensorFlow and JAX are removed.